In [1]:
from janome.tokenizer import Tokenizer

# Tokenizerオブジェクトを作成
tokenizer = Tokenizer()

# 形態素解析を実行
text = "私は犬が好きだ"
tokens = tokenizer.tokenize(text)

# 結果を表示
for token in tokens:
    print(token)

私	名詞,代名詞,一般,*,*,*,私,ワタシ,ワタシ
は	助詞,係助詞,*,*,*,*,は,ハ,ワ
犬	名詞,一般,*,*,*,*,犬,イヌ,イヌ
が	助詞,格助詞,一般,*,*,*,が,ガ,ガ
好き	名詞,形容動詞語幹,*,*,*,*,好き,スキ,スキ
だ	助動詞,*,*,*,特殊・ダ,基本形,だ,ダ,ダ


In [10]:
text = "走っている犬がとても好きだ"
tokens = tokenizer.tokenize(text)

for token in tokens:
    print(f"表層形: {token.surface:<10} 品詞: {token.part_of_speech.split(',')[0]:<6} 原形: {token.base_form}")

表層形: 走っ         品詞: 動詞     原形: 走る
表層形: て          品詞: 助詞     原形: て
表層形: いる         品詞: 動詞     原形: いる
表層形: 犬          品詞: 名詞     原形: 犬
表層形: が          品詞: 助詞     原形: が
表層形: とても        品詞: 副詞     原形: とても
表層形: 好き         品詞: 名詞     原形: 好き
表層形: だ          品詞: 助動詞    原形: だ


In [5]:
def extract_nouns_verbs(text):
    """名詞と動詞だけ原型で取り出す関数"""
    tokens = tokenizer.tokenize(text)
    result = []

    for token in tokens:
        # 品詞を取得(カンマ区切りの最初の要素)
        pos = token.part_of_speech.split(',')[0]
        # 名詞か動詞だけ残す
        if pos in ['名詞', '動詞']:
            result. append(token.base_form)
    return result

text = "走っている犬がとても好きだ"
print(extract_nouns_verbs(text))

['走る', 'いる', '犬', '好き']


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 複数文章を前処理する
corpus = [
    "私は犬が好きだ",
    "猫も犬も好きだ",
    "犬を飼いたいと思っている",
]


# 各文章を形態素解析して空白区切りの文字列に変換する
processed = []
for text in corpus:
    tokens = extract_nouns_verbs(text)
    # TfidVectrizerはスペース区切りを期待するので結合する
    processed.append(' '.join(tokens))

print("前処理後:")
for i, text in enumerate(processed):
    print(f"  文{i+1}: {text}")

# TF-IDFベクトル化
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(processed)

print("\n単語一覧:", tfidf.get_feature_names_out())
print("\n行列:\n", X.toarray().round(2))

前処理後:
  文1: 私 犬 好き
  文2: 猫 犬 好き
  文3: 犬 飼う 思う いる

単語一覧: ['いる' '好き' '思う' '飼う']

行列:
 [[0.   1.   0.   0.  ]
 [0.   1.   0.   0.  ]
 [0.58 0.   0.58 0.58]]


In [8]:
# min_dfとtoken_patternを調整して1文字も含める
tfidf2 = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')
X2 = tfidf2.fit_transform(processed)

print("単語一覧:", tfidf2.get_feature_names_out())
print("\n行列:\n", X2.toarray().round(2))

単語一覧: ['いる' '好き' '思う' '犬' '猫' '私' '飼う']

行列:
 [[0.   0.55 0.   0.43 0.   0.72 0.  ]
 [0.   0.55 0.   0.43 0.72 0.   0.  ]
 [0.55 0.   0.55 0.32 0.   0.   0.55]]
